# FPSA-TRM: Implicit-Differentiation TRM benchmarks (Sudoku-Extreme / Maze-Hard)

Runs the **TRM baseline** vs **Implicit TRM** (fixed-point latent recursion + Neumann implicit gradients, O(1) memory in inner depth) from [`mrinal18/fpsa`](https://github.com/mrinal18/fpsa/tree/fpsa-trm), branch `fpsa-trm`.

**Runtime → Change runtime type → H100 GPU** before starting.

Approximate wall-clock on one H100:
| Stage | Time |
|---|---|
| Dataset build (1k × 1000 aug) | ~5 min |
| Pilot (10% budget), per run | ~40–70 min |
| Full protocol, per run | ~6–10 h |

Everything (datasets, checkpoints, logs) is written to **Google Drive**, and every training cell uses `--resume auto` — if Colab disconnects, just reconnect and re-run the same cell; it continues from the last checkpoint.


In [ ]:
!nvidia-smi


## 1. Google Drive (checkpoints survive disconnects)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fpsa_trm'
import os; os.makedirs(BASE, exist_ok=True)
print('artifacts →', BASE)


## 2. Get the code


In [ ]:
%cd /content
!test -d fpsa || git clone --branch fpsa-trm https://github.com/mrinal18/fpsa.git
%cd /content/fpsa
!git pull --ff-only
!pip -q install pyyaml huggingface_hub pytest
%cd /content/fpsa/experiments/trm


## 3. Sanity check (optional, ~1 min)
34 tests: exact-IFT gradient checks, contraction spectral radii, EMA debiasing, end-to-end training of both models.


In [ ]:
!cd /content/fpsa && python -m pytest tests -q


## 4. Build Sudoku-Extreme (downloads from Hugging Face)
TRM protocol: 1000 training puzzles × 1000 validity-preserving augmentations; full test split (~423k).


In [ ]:
DATA = f'{BASE}/data/sudoku-extreme-1k-aug-1000'
if not os.path.exists(f'{DATA}/train/dataset.json'):
    !python data/build_sudoku_extreme.py --output-dir "{DATA}" --subsample-size 1000 --num-aug 1000
else:
    print('dataset already built')
RESULTS = f'{BASE}/results'


## 5. Pilot runs — 10% budget (~40–70 min each)
Ranks the two models cheaply before committing to full runs. Watch `eval_live/accuracy` for early progress; `eval/exact_accuracy` (EMA, whole-board) stays ~0 until late in training — that is the normal TRM curve shape, not a bug.


In [ ]:
!python train.py --config configs/sudoku_pilot_itrm.yaml \
    --data_dir "{DATA}" --results_dir "{RESULTS}" \
    --run_name pilot-sudoku-itrm-s0 --resume auto


In [ ]:
!python train.py --config configs/sudoku_pilot_trm.yaml \
    --data_dir "{DATA}" --results_dir "{RESULTS}" \
    --run_name pilot-sudoku-trm-s0 --resume auto


## 6. Full protocol (~6–10 h each on H100)
Reference targets: TRM-Att ≈ 75% exact accuracy (MLP variant ≈ 87%). Re-run the cell after any disconnect — `--resume auto` continues from `last.pt`. Checkpoints land every eval (5000 epochs ≈ 6.5k steps); pass `--eval_interval 2500` to checkpoint twice as often.


In [ ]:
!python train.py --config configs/sudoku_itrm.yaml \
    --data_dir "{DATA}" --results_dir "{RESULTS}" \
    --run_name full-sudoku-itrm-s0 --resume auto


In [ ]:
!python train.py --config configs/sudoku_trm.yaml \
    --data_dir "{DATA}" --results_dir "{RESULTS}" \
    --run_name full-sudoku-trm-s0 --resume auto


## 7. Results table + training curves


In [ ]:
!python summarize_results.py --results-dir "{RESULTS}"


In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for run_dir in sorted(glob.glob(f'{RESULTS}/*/')):
    name = os.path.basename(run_dir.rstrip('/'))
    steps, exact, live, iters = [], [], [], []
    for line in open(f'{run_dir}/log.jsonl'):
        r = json.loads(line)
        if 'eval/exact_accuracy' in r:
            steps.append(r['step']); exact.append(r['eval/exact_accuracy'])
            live.append(r.get('eval_live/accuracy', float('nan')))
            iters.append(r.get('eval/inner_iters_per_step', float('nan')))
    if steps:
        axes[0].plot(steps, exact, marker='o', label=name)
        axes[1].plot(steps, live, marker='o', label=name)
        axes[2].plot(steps, iters, marker='o', label=name)
for ax, t in zip(axes, ['eval exact accuracy (EMA)', 'eval cell accuracy (live)', 'inner iterations / step (adaptive compute)']):
    ax.set_title(t); ax.set_xlabel('optimizer step'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 8. Optional: ablations and Maze-Hard
Gradient estimator (neumann / phantom / truncated-BPTT), inner-depth scaling at flat memory (compare `peak_mem_gb` in the logs at `--inner_max_iter 16/32/64`), contractivity mechanisms. Maze-Hard (seq 900) fits an H100-80GB with the config's micro-batching.


In [ ]:
CFG = 'configs/sudoku_pilot_itrm.yaml'
for args, name in [
    ('--grad_mode phantom',            'abl-grad-phantom'),
    ('--grad_mode bptt --bptt_steps 6','abl-grad-bptt6'),
    ('--inner_max_iter 32',            'abl-depth-32'),
    ('--inner_max_iter 64',            'abl-depth-64'),
]:
    !python train.py --config {CFG} {args} \
        --data_dir "{DATA}" --results_dir "{RESULTS}" --run_name {name} --resume auto


In [ ]:
MAZE = f'{BASE}/data/maze-30x30-hard-1k-noaug'
if not os.path.exists(f'{MAZE}/train/dataset.json'):
    !python data/build_maze.py --output-dir "{MAZE}"
!python train.py --config configs/maze_itrm.yaml --data_dir "{MAZE}" \
    --results_dir "{RESULTS}" --run_name maze-itrm-s0 --micro_batch_size 384 --resume auto
!python train.py --config configs/maze_trm.yaml --data_dir "{MAZE}" \
    --results_dir "{RESULTS}" --run_name maze-trm-s0 --micro_batch_size 192 --resume auto
